In [34]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/deepgopal/recipes-with-variation-3/recipes_10000_cleaned_1.csv
/kaggle/input/datasets/deepgopal/recipe-dataset-1/recipes_10000_cleaned.csv
/kaggle/input/datasets/deepgopal/recipes-with-variation/recipes_with_variation.csv


In [35]:
!pip install -q transformers datasets accelerate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [36]:
import pandas as pd
from datasets import Dataset
import random

CSV_PATH = "/kaggle/input/datasets/deepgopal/recipe-dataset-1/recipes_10000_cleaned.csv"
MODEL_NAME = "gpt2"
OUTPUT_DIR = "/kaggle/working/trained_slm_final_3"

df = pd.read_csv(CSV_PATH)

dataset = Dataset.from_pandas(df)

print(dataset.column_names)

['input', 'recipe_name', 'output']


In [37]:
def format_example(example):
    text = example["output"].strip()

    return {"text": text}


dataset = dataset.map(
    format_example,
    remove_columns=dataset.column_names
)

print(dataset[0]["text"])

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

egg yogurt cauliflower spinach okra
<RECIPE_NAME> Egg Curry
<STEP> Heat oil in pan and sauté yogurt until slightly golden.
<STEP> Add cauliflower and cook until the mixture turns soft.
<STEP> Add egg with spinach and stir gently to coat.
<STEP> Cook the mixture until egg absorbs the flavors.
<STEP> Add spices according to your liking and mix well.
<STEP> Simmer on low flame until the curry thickens nicely.
<STEP> Serve hot with rice or roti.
<END>


In [38]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

special_tokens = {
    "additional_special_tokens": [
        "<RECIPE_NAME>", "<STEP>", "<END>"
    ]
}

tokenizer.add_special_tokens(special_tokens)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))


Embedding(50260, 768)

In [39]:
def tokenize(example):

    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


tokenized_dataset = dataset.map(
    tokenize,
    batched=False,
    remove_columns=dataset.column_names
)

print(tokenized_dataset.column_names)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

['input_ids', 'attention_mask', 'labels']


In [40]:
from transformers import default_data_collator

data_collator = default_data_collator

In [41]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=8,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=3e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,

    save_steps=500,
    save_total_limit=2,

    logging_steps=100,

    fp16=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

In [42]:
trainer.train()

Step,Training Loss
100,2.864700
200,0.857300
300,0.504400
400,0.292300
500,0.135900
600,0.083100
700,0.073000
800,0.069500
900,0.067000
1000,0.066200


TrainOutput(global_step=5000, training_loss=0.14633554792404174, metrics={'train_runtime': 5348.5529, 'train_samples_per_second': 14.957, 'train_steps_per_second': 0.935, 'total_flos': 1.045168128e+16, 'train_loss': 0.14633554792404174, 'epoch': 8.0})

In [43]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Training complete.")

Training complete.


In [44]:
import os
import shutil

model_dir = "/kaggle/working/trained_slm_3"

for item in os.listdir(model_dir):
    if item.startswith("checkpoint"):
        shutil.rmtree(os.path.join(model_dir, item))

print("Old checkpoints removed.")

Old checkpoints removed.


In [46]:
!zip -r train_dataset_slm_3 /kaggle/working/trained_slm_final_3

  adding: kaggle/working/trained_slm_3/ (stored 0%)
  adding: kaggle/working/trained_slm_3/model.safetensors

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


^C



zip error: Interrupted (aborting)
